# 08 — Training Run Comparison

Compare multiple training runs side-by-side: reward curves, PPO convergence, summary table.

**Standalone** — runs in its own Colab session; does not need the same runtime as `05_colab_rl_training.ipynb`.  
**Input:** Training artifacts in `MyDrive/optllm_training/training/` (written by notebook 05).  
**Output:** Plots + CSV saved to `MyDrive/optllm_training/analysis/`.

Run all cells top-to-bottom. Only the **Config** cell needs editing.

---
## 1  Config — edit this cell

In [ ]:
# ── Where training artifacts live on Drive ────────────────────────────────────
DRIVE_TRAINING_DIR = '/content/drive/MyDrive/optllm_training'

# ── Which runs to compare ─────────────────────────────────────────────────────
# None → compare ALL runs found; or list specific IDs:
# ['identity_20260504_013045', 'transformer_20260504_020000']
RUN_IDS_TO_COMPARE = None

# ── Repo paths (for importing optimized_llm_planning_memory) ──────────────────
# Option A: clone to /content/ — repos are NOT on Drive (default)
REPO_DIR         = '/content/optimized-llm-planning-memory'
REPO_URL         = 'https://github.com/jiayi-ong/optimized-llm-planning-memory.git'
TRAVEL_WORLD_DIR = '/content/my-travel-world'
TRAVEL_WORLD_URL = 'https://github.com/jiayi-ong/my-travel-world.git'

# Option B: repos live on Drive (uncomment if applicable)
# DRIVE_ROOT       = '/content/drive/MyDrive/STATGR5293 - GenAI Final Project'
# REPO_DIR         = f'{DRIVE_ROOT}/optimized-llm-planning-memory'
# TRAVEL_WORLD_DIR = f'{DRIVE_ROOT}/my-travel-world'

# ── Derived ───────────────────────────────────────────────────────────────────
import sys, os
from pathlib import Path

TRAINING_DIR = Path(DRIVE_TRAINING_DIR) / 'training'
ANALYSIS_DIR = Path(DRIVE_TRAINING_DIR) / 'analysis'
sys.path.insert(0, str(Path(REPO_DIR) / 'src'))

print(f'Training dir : {TRAINING_DIR}')
print(f'Analysis dir : {ANALYSIS_DIR}')

---
## 2  Setup

Mount Drive first, then clone/install the repo so the package can be imported.
No API keys needed — this notebook only reads artifacts, no LLM calls.

In [ ]:
# ── 2a: Mount Drive ───────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

if not TRAINING_DIR.exists():
    print(f'WARNING: {TRAINING_DIR} does not exist yet.')
    print('Run 05_colab_rl_training.ipynb first to produce training artifacts.')
else:
    print(f'Drive mounted. Training dir found: {TRAINING_DIR}')

In [ ]:
# ── 2b: Ensure repo is present and on the right branch ───────────────────────
import subprocess

def _git(args, cwd=None, check=True):
    return subprocess.run(['git'] + args, cwd=cwd, check=check,
                          capture_output=True, text=True)

_use_drive_repo = '/drive/' in REPO_DIR

if _use_drive_repo:
    # Repo lives on Drive — just pull latest, no clone needed
    print(f'Using Drive-mounted repo: {REPO_DIR}')
    _git(['-C', REPO_DIR, 'fetch', 'origin'])
else:
    # Clone to /content/ if not already present
    if not os.path.exists(REPO_DIR):
        assert 'YOUR_ORG' not in REPO_URL, (
            "Update REPO_URL in the Config cell with your actual GitHub org/repo URL, "
            "OR switch to Option B (Drive-mounted repo) by uncommenting those lines."
        )
        _git(['clone', REPO_URL, REPO_DIR])
    else:
        _git(['-C', REPO_DIR, 'fetch', 'origin'])

# Checkout feature branch on whichever repo path we have
_git(['-C', REPO_DIR, 'checkout', 'feat/rl-pipeline-robustness'])
_git(['-C', REPO_DIR, 'pull', 'origin', 'feat/rl-pipeline-robustness'])

# Travel world — same logic
_use_drive_tw = '/drive/' in TRAVEL_WORLD_DIR
if _use_drive_tw:
    print(f'Using Drive-mounted travel world: {TRAVEL_WORLD_DIR}')
    _git(['-C', TRAVEL_WORLD_DIR, 'pull'], check=False)
else:
    if not os.path.exists(TRAVEL_WORLD_DIR):
        assert 'YOUR_ORG' not in TRAVEL_WORLD_URL, (
            "Update TRAVEL_WORLD_URL in the Config cell, or switch to Option B."
        )
        _git(['clone', TRAVEL_WORLD_URL, TRAVEL_WORLD_DIR])
    else:
        _git(['-C', TRAVEL_WORLD_DIR, 'pull'], check=False)

r = _git(['-C', REPO_DIR, 'log', '--oneline', '-1'])
print(f'Repo ready — {r.stdout.strip()}')

In [ ]:
# ── 2c: Install packages (simulator first, then main package) ─────────────────
!pip install -q -e {TRAVEL_WORLD_DIR}
!pip install -q -e {REPO_DIR}[dev]
print('Installation complete.')

---
## 3  Load runs

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

from optimized_llm_planning_memory.training.run_logger import load_episode_metrics, load_ppo_metrics, list_run_ids
from optimized_llm_planning_memory.training.run_manifest import load_manifest, list_manifests

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

available = list_run_ids(TRAINING_DIR)
print(f'Available runs ({len(available)}):')
for rid in available:
    m = load_manifest(rid, TRAINING_DIR)
    label = f'  {rid}'
    if m:
        label += f'  [{m.compressor_type}]  {m.run_name or ""}'
    print(label)

In [ ]:
if RUN_IDS_TO_COMPARE is None:
    RUN_IDS_TO_COMPARE = available

print(f'Comparing {len(RUN_IDS_TO_COMPARE)} run(s): {RUN_IDS_TO_COMPARE}')

In [ ]:
# ── Load episode metrics and manifests for all selected runs ─────────────────

run_data = {}
manifest_data = {}

for run_id in RUN_IDS_TO_COMPARE:
    ep_records = load_episode_metrics(run_id, TRAINING_DIR)
    ppo_records = load_ppo_metrics(run_id, TRAINING_DIR)
    manifest = load_manifest(run_id, TRAINING_DIR)

    if ep_records:
        run_data[run_id] = {
            'ep_df': pd.DataFrame([r.model_dump() for r in ep_records]),
            'ppo_df': pd.DataFrame([r.model_dump() for r in ppo_records]) if ppo_records else pd.DataFrame(),
        }
        manifest_data[run_id] = manifest
        print(f'  {run_id}: {len(ep_records)} episodes, {len(ppo_records)} PPO updates')
    else:
        print(f'  {run_id}: no data (skipping)')

print(f'\nLoaded {len(run_data)} runs with data.')

In [ ]:
# ── Summary table: one row per run ───────────────────────────────────────────

rows = []
for run_id, data in run_data.items():
    ep_df = data['ep_df']
    ppo_df = data['ppo_df']
    manifest = manifest_data.get(run_id)

    last_n = ep_df.tail(50)
    row = {
        'run_id': run_id,
        'compressor_type': manifest.compressor_type if manifest else '?',
        'run_name': manifest.run_name if manifest else '',
        'n_episodes': len(ep_df),
        'final_hard_constraint': last_n['hard_constraint_score'].mean() if 'hard_constraint_score' in last_n else float('nan'),
        'final_soft_constraint': last_n['soft_constraint_score'].mean() if 'soft_constraint_score' in last_n else float('nan'),
        'final_reward_mean': last_n['total_reward'].mean() if 'total_reward' in last_n else float('nan'),
        'final_tool_efficiency': last_n['tool_efficiency_score'].mean() if 'tool_efficiency_score' in last_n else float('nan'),
        'final_success_rate': last_n['success'].mean() if 'success' in last_n else float('nan'),
        'lr': manifest.ppo_hyperparams.get('learning_rate', '?') if manifest else '?',
        'clip_eps': manifest.ppo_hyperparams.get('clip_epsilon', '?') if manifest else '?',
        'ent_coef': manifest.ppo_hyperparams.get('ent_coef', '?') if manifest else '?',
    }
    rows.append(row)

summary_df = pd.DataFrame(rows).sort_values('final_hard_constraint', ascending=False)
print('=== Run Comparison Summary (sorted by final hard constraint score) ===')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
display(summary_df.round(3))

In [ ]:
if not run_data:
    print('No run data to plot.')
else:
    colors = cm.tab10(np.linspace(0, 1, len(run_data)))
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Training Run Comparison — Reward Components', fontsize=14)

    metrics = [
        ('total_reward',              'Total Reward',          axes[0, 0]),
        ('hard_constraint_score',     'Hard Constraint Score', axes[0, 1]),
        ('soft_constraint_score',     'Soft Constraint Score', axes[0, 2]),
        ('tool_efficiency_score',     'Tool Efficiency',       axes[1, 0]),
        ('tool_failure_penalty',      'Tool Failure Penalty',  axes[1, 1]),
        ('logical_consistency_score', 'Logical Consistency',   axes[1, 2]),
    ]

    for col, title, ax in metrics:
        for (run_id, data), color in zip(run_data.items(), colors):
            ep_df = data['ep_df']
            if col not in ep_df.columns:
                continue
            manifest = manifest_data.get(run_id)
            label = (manifest.run_name or run_id[-12:]) if manifest else run_id[-12:]
            window = max(1, len(ep_df) // 20)
            ax.plot(ep_df[col].rolling(window, min_periods=1).mean().values,
                    color=color, linewidth=1.5, label=label)
        ax.set_title(title)
        ax.set_xlabel('Episode')
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    out_path = ANALYSIS_DIR / 'run_comparison.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out_path}')

---
## 4  Reward curves

In [ ]:
# ── PPO convergence comparison ────────────────────────────────────────────────

ppo_runs = {rid: d for rid, d in run_data.items() if not d['ppo_df'].empty}

if ppo_runs:
    colors = cm.tab10(np.linspace(0, 1, len(ppo_runs)))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle('PPO Convergence Diagnostics', fontsize=13)

    for (run_id, data), color in zip(ppo_runs.items(), colors):
        ppo_df = data['ppo_df']
        manifest = manifest_data.get(run_id)
        label = (manifest.run_name if manifest and manifest.run_name else run_id[-8:])

        for ax, col, title in [
            (axes[0], 'approx_kl', 'Approx KL (< 0.02 healthy)'),
            (axes[1], 'clip_fraction', 'Clip Fraction (< 0.2 healthy)'),
            (axes[2], 'explained_variance', 'Explained Variance (→ 1.0)'),
        ]:
            if col in ppo_df.columns:
                ax.plot(ppo_df[col].values, color=color, linewidth=1.2, label=label)
            ax.set_title(title, fontsize=10)
            ax.legend(fontsize=7)
            ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(ANALYSIS_DIR / 'ppo_convergence_comparison.png', dpi=150)
    plt.show()
else:
    print('No PPO update data found in selected runs.')

---
## 5  PPO convergence

In [ ]:
csv_path = ANALYSIS_DIR / 'run_comparison_summary.csv'
summary_df.to_csv(csv_path, index=False)
print(f'Summary CSV saved → {csv_path}')

if not summary_df.empty:
    best = summary_df.iloc[0]
    print(f'\n=== Best run by final hard constraint score ===')
    print(f'  run_id          : {best["run_id"]}')
    print(f'  run_name        : {best["run_name"]}')
    print(f'  compressor_type : {best["compressor_type"]}')
    print(f'  learning_rate   : {best["lr"]}')
    print(f'  clip_epsilon    : {best["clip_eps"]}')
    print(f'  ent_coef        : {best["ent_coef"]}')
    print(f'  hard_constraint : {best["final_hard_constraint"]:.3f}')
    print(f'  reward_mean     : {best["final_reward_mean"]:.3f}')

print(f'\nAll outputs persisted to: {ANALYSIS_DIR}')

---
## 6  Export